# 📝 [제 15편] RNN - 순서를 기억하는 신경망

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「RNN (I) 개념 • 기본구조 • 동작원리」, 「RNN (II) 가중치 • 바이어스 • 동작원리(정량적 분석)」 (ML_LEC_35, 36)

---

## 🎯 학습 목표
1. 순서(sequence)가 있는 데이터의 특징과, 이를 처리하려면 과거 정보를 기억해야 하는 이유를 설명할 수 있다.
2. RNN 의 은닉층 상태 식 Hₜ = tanh(Xₜ · W<sub>ih</sub> + Hₜ₋₁ · W<sub>hh</sub> + b<sub>h</sub>) 을 계산할 수 있다.
3. 간단한 RNN 을 학습시켜 "hello" 의 다음 글자를 예측하고, 기억(순환)이 꼭 필요함을 확인할 수 있다.

---

## 🧭 지난 시간 복습
* 제 14편: CNN 은 사진의 **공간(위치)** 정보를 살렸습니다.
* 오늘은 문장·음성·주가처럼 **순서(시간)** 가 중요한 데이터를 다루는 **RNN (Recurrent Neural Network, 순환 신경망)** 입니다.

[⬅ 이전 : 제 14편 CNN - 합성곱으로 사진의 특징 찾기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/14_cnn.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분 함수 (제 2편, 실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad
print('준비 완료')

## 📖 개념 다지기 1. 순서가 있는 데이터

| 문장 | 뜻 | google 의 품사 |
|:-|:-|:-:|
| I work at **google** | 나는 구글에서 일한다 | 명사 |
| I **google** at work | 나는 회사에서 구글링한다 | 동사 |

같은 단어라도 **놓인 순서** 에 따라 뜻이 달라집니다. 현재 단어의 뜻을 알려면 **앞에 나온 단어들을 기억** 하고 있어야 합니다.

> **[생각해 보기]** 순서가 중요한 데이터의 예를 2가지 더 써 보세요. (                         ,                         )

## 📖 개념 다지기 2. RNN 의 구조 - 은닉층이 자기 자신에게 돌아간다

```
 일반 신경망 :  입력 → [은닉층] → 출력
 RNN       :  입력 → [은닉층] → 출력
                       ↺ (은닉층의 출력이 다음 시간에 다시 은닉층으로 들어감)
```

시간 순서로 펼쳐 그리면

```
  I       work      at       google
  ↓        ↓        ↓          ↓
 [H₁] ──→ [H₂] ──→ [H₃] ──→  [H₄]
  ↓        ↓        ↓          ↓
 출력     출력     출력        출력
```

* H₂ 는 'work' 와 H₁ (= 'I' 의 기억) 을 함께 보고 만들어진다.
* H₃ 는 'at' 과 H₂ (= 'I work' 의 기억) 을 함께 본다.

## 📖 개념 다지기 3. RNN 의 계산 식

> **Hₜ = tanh( Xₜ · W<sub>ih</sub> + Hₜ₋₁ · W<sub>hh</sub> + b<sub>h</sub> )**　　(현재 상태, current state)
> **출력 = softmax( Hₜ · W<sub>ho</sub> + b<sub>o</sub> )**

| 기호 | 뜻 |
|:-|:-|
| Xₜ | 시간 t 의 입력 |
| Hₜ₋₁ | 바로 전 시간의 은닉층 상태 (**기억**) |
| W<sub>ih</sub> | 입력 → 은닉층 가중치 |
| W<sub>hh</sub> | **은닉층 → 은닉층** 가중치 (기억을 얼마나 전달할지) |
| tanh | 활성화 함수 (출력 범위 −1 ~ 1) |
| softmax | 출력들을 **합이 1 인 확률** 로 바꾸는 함수 |

💡 가중치 W<sub>ih</sub>, W<sub>hh</sub>, b<sub>h</sub> 는 **모든 시간에서 같은 값** 을 씁니다.

### [연습 1] 숫자 하나짜리 RNN 손으로 계산하기
W<sub>ih</sub> = 1, W<sub>hh</sub> = 0.5, b<sub>h</sub> = 0, H₀ = 0, 입력 X = (1, 0, 0) (처음에만 신호)
(tanh(1) ≒ 0.76,  tanh(0.38) ≒ 0.36,  tanh(0.18) ≒ 0.18)

| t | Xₜ | Xₜ·1 + Hₜ₋₁·0.5 | Hₜ |
|:-:|:-:|:-:|:-:|
| 1 | 1 | 1 + 0 = 1 | 0.76 |
| 2 | 0 | 0 + 0.76 × 0.5 = ( 　 ) | ( 　 ) |
| 3 | 0 | 0 + ( 　 ) × 0.5 = ( 　 ) | ( 　 ) |

> 입력은 처음에만 있었는데도 H₂, H₃ 가 0 이 아니다! → 은닉층이 첫 입력을 **기억** 하고 있다.
> 만약 W<sub>hh</sub> = 0 이면 H₂ = ( 　 ), H₃ = ( 　 ) → 기억이 ( 남는다 / 사라진다 )

In [ ]:
for Whh in [0.5, 0.0]:
    H = 0.0
    print(f'W_hh = {Whh}')
    for t, x in enumerate([1, 0, 0], start=1):
        H = np.tanh(x*1.0 + H*Whh + 0.0)
        print(f'   t = {t} : 입력 {x} → H = {H:.4f}')

zs = np.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.5)); plt.plot(zs, np.tanh(zs), color='crimson', lw=2, label='tanh')
plt.plot(zs, 1/(1 + np.exp(-zs)), color='gray', ls='--', label='sigmoid (비교)')
plt.axhline(0, color='k', lw=0.6); plt.legend(); plt.grid(alpha=0.3); plt.title('tanh : 출력 −1 ~ 1'); plt.show()

## 🎬 애니메이션 1. 기억이 시간에 따라 전달되는 모습

입력 신호는 **t = 1 에만** 들어갑니다. W<sub>hh</sub> 값에 따라 은닉층의 기억(Hₜ)이 얼마나 오래 남는지 비교하세요.

In [ ]:
#@title 🎬 애니메이션 1 실행 (기억의 전달)
Whhs = [0.0, 0.5, 0.9, 1.5]
Tn = 10
Hs = {}
for w in Whhs:
    H = 0.0; seq = []
    for t in range(Tn):
        H = np.tanh((1.0 if t == 0 else 0.0) + H*w); seq.append(H)
    Hs[w] = seq
frames = [k for k in range(Tn) for _ in range(4)] + [Tn - 1]*8
fig, axes = plt.subplots(len(Whhs), 1, figsize=(11, 6.5))
def draw(f):
    k = frames[f]
    for ax, w in zip(axes, Whhs):
        ax.clear(); ax.set_xlim(-0.6, Tn - 0.4); ax.set_ylim(-0.7, 0.7); ax.axis('off')
        for t in range(Tn):
            v = Hs[w][t] if t <= k else None
            fc = plt.cm.Reds(0.1 + 0.9*v) if v is not None else 'white'
            ax.add_patch(plt.Circle((t, 0), 0.32, fc=fc, ec='k', lw=1.5))
            if v is not None: ax.text(t, 0, f'{v:.2f}', ha='center', va='center', fontsize=9)
            if t < k: ax.annotate('', xy=(t + 0.66, 0), xytext=(t + 0.34, 0), arrowprops=dict(arrowstyle='->', lw=1.5))
        ax.text(-0.55, 0.5, f'W_hh = {w}', fontsize=11, weight='bold')
    axes[0].set_title(f'시간 t = {k+1}  (입력은 t = 1 에만 1, 나머지는 0)', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=180)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** W<sub>hh</sub> = 0 이면 기억이 ( 바로 사라진다 / 오래 남는다 ). W<sub>hh</sub> 가 ( 클수록 / 작을수록 ) 기억이 오래 남는다.
> 💡 RNN 은 학습을 통해 **무엇을 얼마나 기억할지 (W<sub>hh</sub>)** 를 스스로 정합니다.

## 💻 실습. "hello" 를 배우는 RNN

글자 4 종류 h, e, l, o 를 **원-핫 인코딩** 해서 넣고, **다음 글자를 맞히게** 합니다.

| 시간 | 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|:-:|
| 입력 | h | e | l | l |
| 정답 (다음 글자) | e | l | l | o |

> **[생각해 보기]** 시간 3 과 4 의 입력은 똑같이 'l' 인데 정답은 'l' 과 'o' 로 다릅니다. 입력만 보고 답하는 일반 신경망은 이 둘을 구별할 수 ( 있다 / 없다 ). 왜냐하면 (                                   )

학습은 제 2편의 **수치미분 + 경사하강법** 으로 합니다. 손실은 cross-entropy (정답 글자의 확률에 −log).

In [ ]:
chars = ['h', 'e', 'l', 'o']
c2i = {c: i for i, c in enumerate(chars)}
X = np.eye(4)[[c2i[c] for c in 'hell']]          # 입력 원-핫 (4 × 4)
T = [c2i[c] for c in 'ello']                       # 정답 글자 번호
print('입력 원-핫 =\n', X, '\n정답 번호 =', T)

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

class SimpleRNN:
    def __init__(self, hidden=4, use_memory=True):
        self.H = hidden
        self.Wih = np.random.randn(4, hidden) * 0.5           # 입력 → 은닉
        self.Whh = np.random.randn(hidden, hidden) * 0.5 * use_memory   # 은닉 → 은닉 (기억)
        self.bh  = np.zeros(hidden)
        self.Who = np.random.randn(hidden, 4) * 0.5            # 은닉 → 출력
        self.bo  = np.zeros(4)
        self.use_memory = use_memory

    def forward(self):
        h = np.zeros(self.H); states, probs = [], []
        for x in X:
            h = np.tanh(x @ self.Wih + h @ self.Whh + self.bh)  # Hₜ = tanh(Xₜ·Wih + Hₜ₋₁·Whh + bh)
            states.append(h); probs.append(softmax(h @ self.Who + self.bo))
        return np.array(states), np.array(probs)

    def loss(self):
        _, P = self.forward()
        return -sum(np.log(P[t, T[t]] + 1e-7) for t in range(4))

    def train(self, steps=1501, lr=0.1):
        f = lambda x : self.loss()
        params = [self.Wih, self.bh, self.Who, self.bo] + ([self.Whh] if self.use_memory else [])
        self.history = []
        for step in range(steps):
            for P_ in params:
                P_ -= lr * numerical_derivative(f, P_)
            if step % 25 == 0:
                self.history.append((step, self.forward()[1]))
            if step % 500 == 0:
                print(f'   step {step:4d} : 손실 = {self.loss():.4f}')

    def predict(self):
        return ''.join(chars[i] for i in np.argmax(self.forward()[1], axis=1))

for mem in [True, False]:
    np.random.seed(0)
    print('기억 있음 (RNN)' if mem else '기억 없음 (W_hh = 0 고정)')
    model = SimpleRNN(use_memory=mem)
    model.train()
    print('   입력 "hell" → 예측', model.predict(), '\n')
    if mem: rnn = model

> **[확인 1]** 기억이 있는 RNN 의 예측은 ( 　　　　 ), 기억이 없는 모델의 예측은 ( 　　　　 ) 이다.
> 기억이 없는 모델은 세 번째와 네 번째 입력 'l' 에 대해 ( 같은 / 다른 ) 답을 낼 수밖에 없다.

## 🎬 애니메이션 2. RNN 이 다음 글자를 맞혀 가는 학습 과정

입력 h, e, l, l 각각에 대해 다음 글자 확률(h, e, l, o)을 막대로 보여 줍니다. 초록 테두리가 정답입니다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (실습을 먼저 실행하세요)
hist = rnn.history
frames = list(range(len(hist))) + [len(hist) - 1]*8
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
def draw(f):
    step, P = hist[frames[f]]
    for t, ax in enumerate(axes):
        ax.clear()
        cols = ['lightgray']*4; cols[np.argmax(P[t])] = 'royalblue'
        bars = ax.bar(chars, P[t], color=cols, edgecolor=['green' if i == T[t] else 'none' for i in range(4)], linewidth=3)
        ax.set_ylim(0, 1.05)
        ax.set_title(f"입력 '{'hell'[t]}' → 예측 '{chars[np.argmax(P[t])]}'  (정답 '{chars[T[t]]}')", fontsize=11)
    fig.suptitle(f'step {step}', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=150)
plt.close(fig)
show_anim(anim)

> **[관찰 2]** 네 번째 칸(두 번째 'l')에서 'o' 의 확률이 올라가기 시작하는 것은 대략 step ( 　　 ) 부터이다. 이때 RNN 은 "앞에 'l' 이 이미 한 번 나왔다" 는 것을 ( 　　 ) 로 기억하고 있다.

## 📖 개념 다지기 4. RNN 의 한계와 그 다음

* 문장이 길어지면 앞쪽 기억이 점점 흐려집니다 (애니메이션 1 처럼 값이 줄어듦). → **LSTM, GRU** 처럼 기억을 더 잘 지키는 구조가 나왔습니다.
* 요즘의 ChatGPT, Claude 같은 **대규모 언어 모델(LLM)** 은 순환 대신 **어텐션(attention)** 으로 문장 전체를 한꺼번에 보는 **트랜스포머** 구조를 씁니다. 하지만 "앞 단어들을 보고 **다음 단어를 확률로 예측** 한다" 는 생각은 오늘 만든 hello RNN 과 같습니다!

## ✏️ 연습 문제

> **[문제 1]** W<sub>ih</sub> = 2, W<sub>hh</sub> = 1, b<sub>h</sub> = 0, H₀ = 0, 입력 (0.5, −0.5) 일 때 H₁, H₂ 를 구하세요. (tanh(1) ≒ 0.76, tanh(−0.24) ≒ −0.24)
> H₁ = tanh(2 × 0.5 + 0) = ( 　 ),  H₂ = tanh(2 × (−0.5) + 0.76 × 1) = tanh( 　 ) ≒ ( 　 )

> **[문제 2]** 학습된 RNN 으로 다른 입력 순서를 넣어 보세요. 예를 들어 "hel" 만 넣으면 각 시간의 예측은? 아래 코드의 `______` 를 채우세요.

In [ ]:
seq = '______'                                       # 예: 'hel'
h = np.zeros(rnn.H); out = ''
for c in seq:
    x = np.eye(4)[c2i[c]]
    h = np.tanh(x @ rnn.Wih + h @ rnn.Whh + rnn.bh)
    out += chars[np.argmax(h @ rnn.Who + rnn.bo)]
print(f'입력 "{seq}" → 각 시간의 다음 글자 예측 "{out}"')

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : H1 = tanh(1) ≒ 0.76,  H2 = tanh(−1 + 0.76) = tanh(−0.24) ≒ −0.24')
h = np.zeros(rnn.H); out = ''
for c in 'hel':
    h = np.tanh(np.eye(4)[c2i[c]] @ rnn.Wih + h @ rnn.Whh + rnn.bh); out += chars[np.argmax(h @ rnn.Who + rnn.bo)]
print(f'문제 2 : 입력 "hel" → "{out}"  (빈칸 예: hel)')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 순서 데이터 | 문장, 음성, 주가 등 - 앞의 정보가 현재의 의미에 영향 |
| RNN | 은닉층의 출력이 다음 시간의 은닉층으로 다시 들어가는 순환 구조 |
| 상태 식 | Hₜ = tanh(Xₜ · W<sub>ih</sub> + Hₜ₋₁ · W<sub>hh</sub> + b<sub>h</sub>) |
| 기억 | W<sub>hh</sub> 가 과거 정보를 얼마나 전달할지 결정 (학습으로 정해짐) |
| 확인 | 기억이 없으면 "hell → ello" 를 배울 수 없다 |

### 🏆 마무리 도전 과제
제 1편(미분)부터 제 15편(RNN)까지 배운 내용을 한 장의 **마인드맵** 으로 정리해 보세요.
(미분 → 수치미분 → 행렬 → 손실함수 → 경사하강법 → 로지스틱 회귀 → XOR → 신경망 → 오차역전파 → CNN / RNN)

---

[⬅ 이전 : 제 14편 CNN - 합성곱으로 사진의 특징 찾기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/14_cnn.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)